# Step 3: The RAG pipeline
Question, scope from the question, query rewriting, hybrid retrieval, a prompt with numbered passages, a JSON answer with citations, then a figure check: every number in the answer must appear in a cited passage.

In [1]:
import sys, json, pandas as pd
sys.path.insert(0, '../src')
pd.set_option('display.max_colwidth', 120)

In [2]:
from jarvis.rag import RAG
rag = RAG()
out = rag.answer("What was RBC's CET1 ratio at the end of fiscal 2025?")
{k: out[k] for k in ('answer', 'confidence', 'caveats', 'figure_check', 'sources')}

{'answer': "RBC's CET1 ratio at the end of fiscal 2025 was 13.5%.",
 'confidence': 'high',
 'caveats': [],
 'figure_check': [{'figure': '13.5%', 'supported': True}],
 'sources': [{'id': 'S2',
   'bank': 'RBC',
   'document': 'Annual Report',
   'section': 'Regulatory capital and TLAC available | Table 64',
   'year': 2025,
   'period': 'fiscal 2025',
   'page': 191,
   'file': 'd95203dex2.htm'},
  {'id': 'S3',
   'bank': 'RBC',
   'document': 'Annual Report',
   'section': 'Overview',
   'year': 2025,
   'period': 'fiscal 2025',
   'page': 9,
   'file': 'd95203dex2.htm'}]}

In [3]:
out = rag.answer("What was Laurentian Bank's provision for credit losses in fiscal 2025?")
out['answer']

'The provided documents do not contain information about this.'

In [4]:
out = rag.answer("What was BMO's net income in Q2 2025 compared with Q1 2025?")
print(out['scope'])
{k: out[k] for k in ('answer', 'figure_check', 'sources')}

{'banks': ['BMO'], 'years': [2025], 'doc_types': [], 'quarters': ['Q1', 'Q2'], 'quarterly': True}


{'answer': "BMO's reported net income in Q2 2025 was $1,962 million, compared with $2,138 million in Q1 2025, a decrease of $176 million. Adjusted net income in Q2 2025 was $2,046 million, compared with $2,289 million in Q1 2025, a decrease of $243 million.",
 'figure_check': [{'figure': '$1,962 million', 'supported': True},
  {'figure': '$2,138 million', 'supported': True},
  {'figure': '$176 million',
   'supported': True,
   'computed': '$1,962 million minus $2,138 million'},
  {'figure': '$2,289 million', 'supported': True},
  {'figure': '$243 million', 'supported': False}],
 'sources': [{'id': 'S1',
   'bank': 'BMO',
   'document': 'Quarterly Report',
   'section': 'Report To Shareholders',
   'year': 2025,
   'period': 'Q1 fiscal 2025',
   'page': 1,
   'file': 'Q1.pdf'},
  {'id': 'S2',
   'bank': 'BMO',
   'document': 'Quarterly Report',
   'section': 'Report To Shareholders',
   'year': 2025,
   'period': 'Q2 fiscal 2025',
   'page': 1,
   'file': 'Q2.pdf'}]}

## The 15 compliance questions (Step 3)
All five banks, three questions each, across credit, market, liquidity, operational, climate and capital risk, annual and quarterly. The true answer to each was located in the filing by hand **before** any run, with its pages (`reports/step3_questions.json`). For each answer we record the retrieved chunks, the answer, the cited pages, whether the key figure is present and whether a citation lands on a page holding the truth; then each answer was read against its truth and cited passages and given a manual verdict (`reports/step3_review.json`).

In [5]:
qs = pd.DataFrame(json.load(open('../reports/step3_questions.json')))
qs[['id', 'bank', 'risk', 'period', 'question', 'truth', 'where']]

,id,bank,risk,period,question,truth,where
0,Q01,RBC,credit,fiscal 2025,What was RBC's total provision for credit losses in fiscal 2025?,"$4,362 million, up $1,130 million or 35% from fiscal 2024",RBC_2025_AR p13 and p15
1,Q02,RBC,liquidity,Q3 2025,What was RBC's average liquidity coverage ratio for the third quarter of fiscal 2025?,"129% (surplus of about $103 billion), down from 131% in the prior quarter",RBC_2025_Q3 p38
2,Q03,RBC,climate,fiscal 2025,How does RBC manage climate-related risk according to its fiscal 2025 annual report?,"Climate and environmental and social risk is treated as transverse, affecting principal risks (strategic, operationa...",RBC_2025_AR p177 and p181
3,Q04,TD,credit,Q2 2025,What was TD's provision for credit losses in Q2 2025?,"$1,341 million, up $129 million from the prior quarter",TD_2025_Q2 p15
4,Q05,TD,liquidity,fiscal 2025,"What was TD's average liquidity coverage ratio for the quarter ended October 31, 2025?","130% for the quarter ended October 31, 2025 (fourth quarter, reported in the fiscal 2025 MD&A)",TD_2025_MDA p92
5,Q06,TD,operational,fiscal 2025,What does TD say about technology and cybersecurity risk in its fiscal 2025 MD&A?,"Technology and cybersecurity risks continue to increase, especially with heightened geopolitical tensions and a chal...",TD_2025_MDA p60 and p61
6,Q07,BMO,credit,Q2 2025,What was BMO's total provision for credit losses in Q2 2025?,"$1,054 million ($765 million on impaired loans and $289 million on performing loans)",BMO_2025_Q2 p6
7,Q08,BMO,capital,Q3 2025,What was BMO's CET1 ratio at the end of Q3 2025?,"13.5% as at July 31, 2025, relatively unchanged from Q2 2025",BMO_2025_Q3 p18
8,Q09,BMO,climate,fiscal 2024,How did BMO describe its exposure to and management of climate risk in fiscal 2024?,Exposure to physical and transition risks; climate risk could disrupt operations and affect customers and counterpar...,"BMO_2024_MDA p115, p189 and p191"
9,Q10,BNS,capital,Q2 2025,"What was Scotiabank's CET1 capital ratio as at April 30, 2025?","13.2%, up about 30 basis points from the prior quarter",BNS_2025_Q2 p53


In [6]:
res = pd.DataFrame(json.load(open('../reports/step3_results.json')))
rev = pd.DataFrame(json.load(open('../reports/step3_review.json'))['reviews'])
CFG = 'native:cot+v5'   # the final configuration; earlier runs are compared below
best = res[res.config == CFG].merge(rev[rev.config == CFG], on=['id', 'config'])
best['retrieved'] = best.retrieved.map(lambda ids: ', '.join(i.split('#')[0] + '#' + i.split('#')[1] for i in ids[:3]) + ' ...')
best[['id', 'answer', 'retrieved', 'key_found', 'cites_truth_page', 'verdict', 'citations_accurate', 'note']]

,id,answer,retrieved,key_found,cites_truth_page,verdict,citations_accurate,note
0,Q01,"RBC's total provision for credit losses in fiscal 2025 was $4,362 million [S7].","RBC_2025_AR#0179, RBC_2025_AR#0000, RBC_2025_AR#0019 ...",True,False,correct,True,"$4,362 million, cited to the PCL table on p223 (the same table also appears on p13)"
1,Q02,"RBC's average liquidity coverage ratio (LCR) for the third quarter of fiscal 2025 (quarter ended July 31, 2025) was ...","RBC_2025_Q3#0075, RBC_2025_Q3#0000, RBC_2025_Q3#0002 ...",True,True,correct,True,"129%, average for the quarter ended July 31, 2025, p38"
2,Q03,"According to RBC's fiscal 2025 annual report, the bank manages climate-related risk through several key approaches: ...","RBC_2025_AR#0142, RBC_2025_AR#0146, RBC_2025_AR#0136 ...",None,True,correct,True,"integration into risk management, scenario analysis, enterprise stress testing, transition and physical risk; ground..."
3,Q04,"TD's provision for credit losses in Q2 2025 was $1,341 million [S1]. This comprised provision for credit losses - St...","TD_2025_Q2#0026, TD_2025_Q2#0066, TD_2025_Q2#0154 ...",True,True,correct,True,"$1,341 million with the Stage 3 and Stage 1 and 2 split, p15"
4,Q05,"TD's average liquidity coverage ratio for the quarter ended October 31, 2025 was 130%.","TD_2025_MDA#0189, TD_2025_MDA#0226, TD_2025_MDA#0000 ...",True,True,correct,True,"130% for the quarter ended October 31, 2025, p92"
5,Q06,TD describes technology and cybersecurity risk as follows: The Bank manages technology and cybersecurity risks to su...,"TD_2025_MDA#0128, TD_2025_MDA#0011, TD_2025_MDA#0179 ...",None,True,correct,True,"technology and cyber risk governance (Platforms and Technology Risk Oversight Committee, enterprise programs) from p..."
6,Q07,"BMO's total provision for credit losses in Q2 2025 was $1,054 million. [S1] This represented 63 basis points as a pe...","BMO_2025_Q2#0033, BMO_2025_Q2#0071, BMO_2025_Q2#0031 ...",True,False,correct,True,"$1,054 million ($765 million impaired, $289 million performing); the chunk starting on p14 holds the figure"
7,Q08,"BMO's Common Equity Tier 1 (CET1) Ratio was 13.5% as at July 31, 2025 (the end of Q3 fiscal 2025).","BMO_2025_Q3#0037, BMO_2025_Q3#0000, BMO_2025_Q3#0082 ...",True,True,correct,True,"13.5% as at July 31, 2025; the cited chunk runs from p17 to p18 where the sentence is"
8,Q09,BMO described its exposure to and management of climate risk in fiscal 2024 through several integrated approaches: [...,"BMO_2024_MDA#0067, BMO_2024_MDA#0171, BMO_2024_MDA#0056 ...",None,True,correct,True,"physical and transition risk, channels into credit, market, operational and liquidity risk, risk appetite and sector..."
9,Q10,"Scotiabank's CET1 capital ratio was 13.2% as at April 30, 2025.","BNS_2025_Q2#0122, BNS_2025_Q2#0121, BNS_2025_Q2#0085 ...",True,True,correct,True,"13.2% as at April 30, 2025 (after the date fix), cited to the CET1 sentence on p53"


In [7]:
acc = (best.verdict == 'correct').mean()
part = (best.verdict == 'partly correct').mean()
cit = best.citations_accurate.mean()
print(f'Accuracy (manual): {acc:.0%} correct, {part:.0%} partly correct; citations accurate: {cit:.0%}')

Accuracy (manual): 100% correct, 0% partly correct; citations accurate: 100%


### Failure analysis
The final configuration answers all 15 correctly. Each earlier run's failures led to one logged mechanism fix: a quarter named by its end date (scope rule), and a table separated from the paragraph that explains it (neighbouring chunks).

In [8]:
progress = rev.groupby('config', sort=False).verdict.apply(lambda v: f"{(v == 'correct').sum()} of {len(v)} correct")
progress[[c for c in ('native:cot', 'native:cot+datefix', 'native:cot+v4', 'native:cot+v5') if c in progress.index]]

config
native:cot            13 of 15 correct
native:cot+datefix    14 of 15 correct
native:cot+v4         15 of 15 correct
native:cot+v5         15 of 15 correct
Name: verdict, dtype: str

In [9]:
first = rev[rev.config.isin(['native:cot', 'native:cot+datefix'])]
first[first.verdict != 'correct'][['config', 'id', 'verdict', 'note']]

,config,id,verdict,note
39,native:cot,Q10,wrong,"declined: the question names the quarter by its end date (April 30, 2025); the scope rule did not read dates, so onl..."
43,native:cot,Q14,partly correct,states the +$0.4 million change correctly but says the level is not disclosed: the retrieved chunk holds the explana...
73,native:cot+datefix,Q14,partly correct,states the +$0.4 million change correctly but says the level is not disclosed: the retrieved chunk holds the explana...
